<a href="https://colab.research.google.com/github/SAMYSOSERIOUS/Master-Thesis/blob/main/notebooks_standardized/SOTA/nhanes_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 · Setup

In [ ]:
import subprocess, sys, importlib
subprocess.run([sys.executable,'-m','pip','install','-q','cleanlab'], check=False)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, json, gc, re, math, random
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
PROJECT = Path('/content/drive/MyDrive/Master Thesis')
sys.path.insert(0, str(PROJECT/'scope3'))
import config; importlib.reload(config)
import training_lib_max as TM
device = 'cuda' if torch.cuda.is_available() else 'cpu'; assert device=='cuda', 'Switch to a GPU runtime.'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('TM loaded:', hasattr(TM,'prepare_local_data'), '| GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
TM loaded: True | GPU: Tesla T4


## 2 · Config

In [ ]:
OUTDIR = PROJECT/'nhanes_benchmark'; CKPT = OUTDIR/'ckpts'
for d in [OUTDIR, CKPT]: d.mkdir(parents=True, exist_ok=True)
BACKBONE = 'small'
RES = 224
CLEAN_MODE = 'none'
CAP_CW = 6.0
SPLIT_SEED = 0
TEST_FRAC, VAL_FRAC_OF_REST = 0.15, 0.1765
EPOCHS, PATIENCE, MIN_EPOCHS, LR = 40, 8, 8, 3e-4
print(f'BACKBONE={BACKBONE} · CLEAN_MODE={CLEAN_MODE} · RES={RES}')

BACKBONE=small · CLEAN_MODE=none · RES=224


## 3 · Training core (RAM-array dataset, EMA, discriminative-LR, TTA, robust folds)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score, recall_score
from sklearn.model_selection import train_test_split, GroupKFold

class ArrayDS(Dataset):
    def __init__(self, df, images, train=False, size=RES):
        self.df=df.reset_index(drop=True); self.images=images; self.train=train
        self.aug=(T.Compose([T.RandomHorizontalFlip(0.5), T.RandomRotation(12),
                             T.RandomResizedCrop(size,scale=(0.85,1.0),antialias=True),
                             T.RandomApply([T.ColorJitter(0.2,0.2)],p=0.3)]) if train
                  else T.Compose([T.Resize((size,size),antialias=True)]))
        self.norm=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; arr=np.asarray(self.images[int(r['arr_idx'])]).astype(np.float32)/255.0
        x=torch.from_numpy(arr).unsqueeze(0).repeat(3,1,1)
        return self.norm(self.aug(x)), int(r['kl_grade'])

def build_convnext(arch='small', device='cuda'):
    if arch=='tiny':   m=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    elif arch=='base': m=models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
    else:              m=models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
    m.classifier[2]=nn.Linear(m.classifier[2].in_features,5); return m.to(device)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay; self.shadow={k:v.detach().clone() for k,v in model.state_dict().items()}
    def update(self, model):
        for k,v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
            else: self.shadow[k]=v.detach().clone()
    def state(self): return {k:v.clone() for k,v in self.shadow.items()}

@torch.no_grad()
def predict(model, ds_maker, df, device='cuda', bs=64, tta=False):
    model.eval(); out=[]
    ld=DataLoader(ds_maker(df,train=False), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    for x,_ in ld:
        x=x.to(device,non_blocking=True)
        with torch.amp.autocast('cuda'):
            p=model(x).softmax(1)
            if tta: p=p+model(torch.flip(x,[3])).softmax(1)
        out.append((p/(2 if tta else 1)).float().cpu().numpy())
    return np.concatenate(out)

def train_generic(train_df,val_df,*,build_fn,ds_maker,class_weights=None,epochs=40,patience=8,
                  min_epochs=1,bs=32,lr=3e-4,warmup=3,seed=0,device='cuda',log=print):
    torch.manual_seed(seed)
    tr=DataLoader(ds_maker(train_df,train=True), batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    model=build_fn(device=device)
    head=[p for n,p in model.named_parameters() if 'classifier' in n]
    body=[p for n,p in model.named_parameters() if 'classifier' not in n]
    opt=torch.optim.AdamW([{'params':body,'lr':lr*0.1},{'params':head,'lr':lr}], weight_decay=1e-4)
    from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
    sched=SequentialLR(opt,[LinearLR(opt,0.1,1.0,total_iters=warmup),CosineAnnealingLR(opt,T_max=max(1,epochs-warmup))],milestones=[warmup])
    cw=torch.tensor(class_weights,dtype=torch.float32,device=device) if class_weights is not None else None
    crit=nn.CrossEntropyLoss(weight=cw,label_smoothing=0.05); scaler=torch.amp.GradScaler('cuda'); ema=EMA(model,0.999)
    best=-1; best_state=None; wait=0; yv=val_df['kl_grade'].values
    for ep in range(1,epochs+1):
        model.train(); opt.zero_grad(set_to_none=True)
        for x,y in tr:
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            with torch.amp.autocast('cuda'): loss=crit(model(x),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(model)
        sched.step()
        tmp=build_fn(device=device); tmp.load_state_dict(ema.state()); pv=predict(tmp,ds_maker,val_df,device); del tmp
        acc=accuracy_score(yv,pv.argmax(1)); qwk=cohen_kappa_score(yv,pv.argmax(1),weights='quadratic')
        log(f'  ep{ep:02d}/{epochs} val_acc={acc:.4f} qwk={qwk:.3f}'+('  *best*' if acc>best+1e-4 else f'  ({wait+1}/{patience})'))
        if acc>best+1e-4: best=acc; best_state=ema.state(); wait=0
        else:
            wait+=1
            if wait>=patience and ep>=min_epochs: log(f'  early stop ep{ep}, best={best:.4f}'); break
    del model,tr; gc.collect(); torch.cuda.empty_cache()
    return best_state,best

def robust_fold(train_df,val_df,*,min_quality=0.40,**kw):
    log=kw.get('log',print)
    st,best=train_generic(train_df,val_df,seed=kw.pop('seed',0),min_epochs=6,**kw)
    if best<min_quality:
        log(f'  WEAK FOLD (best={best:.3f}) — retry...'); st,best=train_generic(train_df,val_df,seed=999,min_epochs=6,**kw)
    return st,best
print('Core ready.')

Core ready.


## 4 · Stage 1 — load NHANES from the packed array

In [ ]:
manifest = TM.prepare_local_data(); images = TM._IMAGES
cands = [d for d in manifest['dataset'].unique() if 'nhan' in str(d).lower()]
assert cands, f'No NHANES dataset found. Keys present: {list(manifest["dataset"].unique())}'
KEY = cands[0]
df = manifest[manifest['dataset']==KEY].copy().reset_index(drop=True)
df['orig_kl'] = df['kl_grade'].astype(int)
# patient grouping if available, else image-level
if 'patient_id' in df.columns: df['grp'] = df['patient_id'].astype(str)
elif 'filename' in df.columns:
    ext = df['filename'].astype(str).str.extract(r'(\d{4,})')[0]
    df['grp'] = ext.fillna(pd.Series(df.index.astype(str)))
else: df['grp'] = df.index.astype(str)
print(f'NHANES key: {KEY} | rows: {len(df)} | groups: {df["grp"].nunique()}')
print('KL distribution:', dict(df['orig_kl'].value_counts().sort_index()))
print('KL %:', {k:f'{v*100:.1f}%' for k,v in df['orig_kl'].value_counts(normalize=True).sort_index().items()})
mkNH = lambda d,train: ArrayDS(d, images, train=train, size=RES)

Copied array in 49s
Loaded array (61558, 224, 224) in 13s
NHANES key: nhanes3 | rows: 4785 | groups: 2412
KL distribution: {0: np.int64(2563), 1: np.int64(546), 2: np.int64(1271), 3: np.int64(307), 4: np.int64(98)}
KL %: {0: '53.6%', 1: '11.4%', 2: '26.6%', 3: '6.4%', 4: '2.0%'}


## 5 · Stage 2 — fixed split (stratified, saved, leak-asserted)

In [ ]:
SPLIT_JSON = OUTDIR/'nhanes_split.json'
if SPLIT_JSON.exists():
    sp=json.loads(SPLIT_JSON.read_text()); TRAIN_G,VAL_G,TEST_G=set(sp['train']),set(sp['val']),set(sp['test'])
    print('loaded split.')
else:
    g = df.groupby('grp')['orig_kl'].max().reset_index().rename(columns={'orig_kl':'pmax'}).sort_values('grp').reset_index(drop=True)
    def safe(frame,frac,seed):
        strat = frame['pmax'] if (frame['pmax'].value_counts()>=2).all() else None
        return train_test_split(frame,test_size=frac,random_state=seed,stratify=strat)
    trv,te=safe(g,TEST_FRAC,SPLIT_SEED); tr,va=safe(trv,VAL_FRAC_OF_REST,SPLIT_SEED)
    TRAIN_G,VAL_G,TEST_G=set(tr['grp']),set(va['grp']),set(te['grp'])
    SPLIT_JSON.write_text(json.dumps({'train':sorted(TRAIN_G),'val':sorted(VAL_G),'test':sorted(TEST_G)}))
    print('created + saved split.')
for a,b in [(TRAIN_G,VAL_G),(TRAIN_G,TEST_G),(VAL_G,TEST_G)]: assert a.isdisjoint(b),'LEAK!'
df['split']=np.where(df.grp.isin(TEST_G),'test',np.where(df.grp.isin(VAL_G),'val','train'))
print('knees per split:', dict(df['split'].value_counts()))

created + saved split.
knees per split: {'train': np.int64(3350), 'test': np.int64(718), 'val': np.int64(717)}


## 6 · Stage 3 (optional) — CleanLab-corrected labels (only if CLEAN_MODE='aggressive')

In [ ]:
df['clean_kl'] = df['orig_kl']
if CLEAN_MODE == 'aggressive':
    from cleanlab.filter import find_label_issues
    labels = df['orig_kl'].values.astype(int); oos = np.zeros((len(df),5),dtype=np.float32)
    freq = df['orig_kl'].value_counts().sort_index()
    cwc = np.minimum(len(df)/(5.0*np.array([freq.get(k,1) for k in range(5)])), CAP_CW); cwc=(cwc/cwc.mean()).tolist()
    for fi,(tri,vai) in enumerate(GroupKFold(3).split(df, groups=df['grp'].values)):
        print(f'clean fold {fi+1}/3'); st,_=robust_fold(df.iloc[tri],df.iloc[vai],build_fn=lambda device:build_convnext(BACKBONE,device),
             ds_maker=mkNH,class_weights=cwc,epochs=16,patience=4,bs=32,lr=LR,device=device,log=print)
        m=build_convnext(BACKBONE,device); m.load_state_dict(st); oos[vai]=predict(m,mkNH,df.iloc[vai],device); del m; gc.collect(); torch.cuda.empty_cache()
    flagged=find_label_issues(labels=labels,pred_probs=oos,return_indices_ranked_by='self_confidence')
    alt=oos.argmax(1); corr=labels.copy()
    for i in flagged: corr[i]=alt[i]
    df['clean_kl']=corr; print(f'relabeled {int((corr!=labels).sum())} ({100*(corr!=labels).mean():.1f}%)')
else:
    print('CLEAN_MODE=none -> original labels only (the honest first-benchmark number).')

CLEAN_MODE=none -> original labels only (the honest first-benchmark number).


## 7 · Stage 4 — train + TTA

In [ ]:
tr_df=df[df.split=='train'].copy(); va_df=df[df.split=='val'].copy(); te_df=df[df.split=='test'].copy()
lblcol = 'clean_kl' if CLEAN_MODE!='none' else 'orig_kl'
tr_df['kl_grade']=tr_df[lblcol].astype(int); va_df['kl_grade']=va_df[lblcol].astype(int)
fr=pd.Series(tr_df['kl_grade']).value_counts().sort_index()
cw=np.minimum(len(tr_df)/(5.0*np.array([fr.get(k,1) for k in range(5)])), CAP_CW); cw=(cw/cw.mean()).tolist()
print(f'train {len(tr_df)} | val {len(va_df)} | test {len(te_df)} | capped class weights {[round(x,2) for x in cw]}')

ck = CKPT/f'convnext_{BACKBONE}_{CLEAN_MODE}.pt'
if ck.exists():
    print('checkpoint found -> loading'); st=torch.load(ck,map_location=device)
else:
    st,best=train_generic(tr_df,va_df,build_fn=lambda device:build_convnext(BACKBONE,device),ds_maker=mkNH,
                          class_weights=cw,epochs=EPOCHS,patience=PATIENCE,min_epochs=MIN_EPOCHS,bs=32,lr=LR,device=device,log=print)
    torch.save(st,ck); print(f'saved -> {ck}  (best val_acc {best:.4f})')
model=build_convnext(BACKBONE,device); model.load_state_dict(st); model.eval()
probs=predict(model,mkNH,te_df,device,tta=True); pred=probs.argmax(1)

train 3350 | val 717 | test 718 | capped class weights [0.15, 0.74, 0.32, 1.3, 2.49]
Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:01<00:00, 125MB/s]


  ep01/40 val_acc=0.0251 qwk=0.004  *best*
  ep02/40 val_acc=0.0279 qwk=0.008  *best*
  ep03/40 val_acc=0.0404 qwk=0.015  *best*
  ep04/40 val_acc=0.0823 qwk=0.033  *best*
  ep05/40 val_acc=0.1283 qwk=0.103  *best*
  ep06/40 val_acc=0.2162 qwk=0.190  *best*
  ep07/40 val_acc=0.3026 qwk=0.267  *best*
  ep08/40 val_acc=0.3612 qwk=0.325  *best*
  ep09/40 val_acc=0.4031 qwk=0.395  *best*
  ep10/40 val_acc=0.4365 qwk=0.457  *best*
  ep11/40 val_acc=0.4435 qwk=0.511  *best*
  ep12/40 val_acc=0.4589 qwk=0.551  *best*
  ep13/40 val_acc=0.4826 qwk=0.574  *best*
  ep14/40 val_acc=0.5035 qwk=0.602  *best*
  ep15/40 val_acc=0.5216 qwk=0.640  *best*
  ep16/40 val_acc=0.5384 qwk=0.642  *best*
  ep17/40 val_acc=0.5523 qwk=0.663  *best*
  ep18/40 val_acc=0.5509 qwk=0.662  (1/8)
  ep19/40 val_acc=0.5565 qwk=0.667  *best*
  ep20/40 val_acc=0.5662 qwk=0.672  *best*
  ep21/40 val_acc=0.5690 qwk=0.676  *best*
  ep22/40 val_acc=0.5718 qwk=0.680  *best*
  ep23/40 val_acc=0.5788 qwk=0.697  *best*
  ep24/40 va

## 8 · Stage 5 — results (imbalance-aware)

In [ ]:
yo=te_df['orig_kl'].values.astype(int)
acc=accuracy_score(yo,pred); bacc=balanced_accuracy_score(yo,pred); qwk=cohen_kappa_score(yo,pred,weights='quadratic')
n=len(yo); ci=1.96*math.sqrt(acc*(1-acc)/n); rec=recall_score(yo,pred,average=None,labels=[0,1,2,3,4],zero_division=0)

print('='*58); print(f'NHANES III · ConvNeXt-{BACKBONE} @ {RES} + TTA · first benchmark'); print('='*58)
print(f'  overall accuracy (original) : {acc*100:.2f}%  ±{ci*100:.1f}   (n={n})')
print(f'  balanced accuracy           : {bacc*100:.2f}%   (mean per-class recall)')
print(f'  QWK                         : {qwk:.3f}')
print(f'  per-class recall            : ' + '  '.join(f'KL{k}={rec[k]*100:.0f}%' for k in range(5)))
if CLEAN_MODE!='none':
    yc=te_df['clean_kl'].values.astype(int)
    print(f'  corrected-label accuracy    : {accuracy_score(yc,pred)*100:.2f}%   (CleanLab {CLEAN_MODE}, secondary)')
print('\nconfusion matrix (rows = true KL 0..4):'); print(confusion_matrix(yo,pred,labels=[0,1,2,3,4]))

json.dump({'dataset':'NHANES III','backbone':f'convnext_{BACKBONE}','clean_mode':CLEAN_MODE,
           'acc_original':float(acc),'balanced_acc':float(bacc),'qwk':float(qwk),'ci95':float(ci),'n':int(n),
           'per_class_recall':[float(r) for r in rec]}, open(OUTDIR/'nhanes_benchmark_summary.json','w'), indent=2)
print(f'\nsaved -> {OUTDIR/"nhanes_benchmark_summary.json"}')

NHANES III · ConvNeXt-small @ 224 + TTA · first benchmark
  overall accuracy (original) : 62.12%  ±3.5   (n=718)
  balanced accuracy           : 51.46%   (mean per-class recall)
  QWK                         : 0.715
  per-class recall            : KL0=72%  KL1=37%  KL2=59%  KL3=47%  KL4=43%

confusion matrix (rows = true KL 0..4):
[[269  68  34   2   1]
 [ 35  30  16   0   0]
 [ 28  30 120  24   2]
 [  0   1  17  21   6]
 [  0   0   1   7   6]]

saved -> /content/drive/MyDrive/Master Thesis/nhanes_benchmark/nhanes_benchmark_summary.json
